# 🐼 Pandas — From Basics to Advanced

Pandas is Python's primary library for working with **tabular data** (rows & columns, like a
spreadsheet or SQL table). Its two core objects:
- **`Series`** — a single labeled 1D column of data
- **`DataFrame`** — a 2D table made of multiple `Series` sharing an index

## Table of Contents
1. [Series & DataFrames: creation](#1)
2. [Reading & writing files (CSV/JSON)](#2)
3. [Inspecting data](#3)
4. [Selecting & filtering](#4)
5. [Adding, modifying, dropping columns](#5)
6. [Handling missing data](#6)
7. [GroupBy & aggregation](#7)
8. [Merging & joining](#8)
9. [Advanced: apply, pivot tables, time series](#9)

In [1]:
import pandas as pd
import numpy as np
print(pd.__version__)


3.0.3


In [2]:
# =========================================================
# SERIES - a single labeled column
# =========================================================

s = pd.Series([10, 20, 30, 40], index=["a", "b", "c", "d"])
print(s)
print(s["b"])        # label-based access
print(s.values)       # underlying numpy array
print(s.index)        # the labels


a    10
b    20
c    30
d    40
dtype: int64
20
[10 20 30 40]
Index(['a', 'b', 'c', 'd'], dtype='str')


In [3]:
# =========================================================
# DATAFRAME - the main object you'll use constantly
# =========================================================

# Most common way: from a dict of lists (keys become column names)
df = pd.DataFrame({
    "name": ["Alice", "Bob", "Charlie", "Diana"],
    "age": [25, 30, 35, 28],
    "city": ["NYC", "LA", "NYC", "Chicago"],
    "salary": [70000, 85000, 62000, 90000],
})
print(df)

# From a list of dicts (rows) - equally common, e.g. when reading from an API
rows = [
    {"name": "Eve", "age": 22},
    {"name": "Frank", "age": 40},
]
print(pd.DataFrame(rows))

      name  age     city  salary
0    Alice   25      NYC   70000
1      Bob   30       LA   85000
2  Charlie   35      NYC   62000
3    Diana   28  Chicago   90000
    name  age
0    Eve   22
1  Frank   40


In [4]:
# =========================================================
# READING / WRITING CSV, JSON, EXCEL
# These are the #1 most common way real data enters/leaves a pandas workflow.
# =========================================================

df.to_csv("people.csv", index=False)         # index=False -> don't write the row index as a column
loaded = pd.read_csv("people.csv")
print(loaded)

df.to_json("people.json", orient="records", indent=2)
loaded_json = pd.read_json("people.json")
print(loaded_json)

# df.to_excel("people.xlsx", index=False)     # needs openpyxl installed
# pd.read_excel("people.xlsx")

# pd.read_csv also has many useful options, e.g.:
# pd.read_csv("file.csv", usecols=["name", "age"], dtype={"age": int}, na_values=["N/A"])


      name  age     city  salary
0    Alice   25      NYC   70000
1      Bob   30       LA   85000
2  Charlie   35      NYC   62000
3    Diana   28  Chicago   90000
      name  age     city  salary
0    Alice   25      NYC   70000
1      Bob   30       LA   85000
2  Charlie   35      NYC   62000
3    Diana   28  Chicago   90000


In [5]:
# =========================================================
# INSPECTING A DATAFRAME - always the first thing to do with new data
# =========================================================

print(df.head(2))        # first 2 rows
print(df.tail(2))        # last 2 rows
print(df.shape)          # (rows, columns)
print(df.columns.tolist())
print(df.dtypes)         # data type of each column
print(df.info())         # summary: dtypes, non-null counts, memory usage
print(df.describe())     # summary STATISTICS for numeric columns (mean, std, quartiles...)

    name  age city  salary
0  Alice   25  NYC   70000
1    Bob   30   LA   85000
      name  age     city  salary
2  Charlie   35      NYC   62000
3    Diana   28  Chicago   90000
(4, 4)
['name', 'age', 'city', 'salary']
name        str
age       int64
city        str
salary    int64
dtype: object
<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   name    4 non-null      str  
 1   age     4 non-null      int64
 2   city    4 non-null      str  
 3   salary  4 non-null      int64
dtypes: int64(2), str(2)
memory usage: 295.0 bytes
None
             age        salary
count   4.000000      4.000000
mean   29.500000  76750.000000
std     4.203173  12996.794477
min    25.000000  62000.000000
25%    27.250000  68000.000000
50%    29.000000  77500.000000
75%    31.250000  86250.000000
max    35.000000  90000.000000


In [6]:
# =========================================================
# SELECTING COLUMNS
# =========================================================

print(df["name"])              # single column -> returns a Series
print(df[["name", "age"]])     # multiple columns -> returns a DataFrame (note the double brackets)

0      Alice
1        Bob
2    Charlie
3      Diana
Name: name, dtype: str
      name  age
0    Alice   25
1      Bob   30
2  Charlie   35
3    Diana   28


In [7]:
# =========================================================
# SELECTING ROWS - .loc (label-based) vs .iloc (integer position-based)
# =========================================================

print(df.loc[0])                    # row with index label 0
print(df.loc[0:2])                  # rows 0 through 2 INCLUSIVE (label slicing includes the end!)
print(df.loc[0, "name"])            # specific cell: row 0, column 'name'
print(df.loc[df["age"] > 28, "name"])   # boolean filter + column selection combined

print(df.iloc[0])                   # first row, by POSITION
print(df.iloc[0:2])                 # first 2 rows, position slicing EXCLUDES the end (like normal Python)
print(df.iloc[0, 1])                # row 0, column at position 1


name      Alice
age          25
city        NYC
salary    70000
Name: 0, dtype: object
      name  age city  salary
0    Alice   25  NYC   70000
1      Bob   30   LA   85000
2  Charlie   35  NYC   62000
Alice
1        Bob
2    Charlie
Name: name, dtype: str
name      Alice
age          25
city        NYC
salary    70000
Name: 0, dtype: object
    name  age city  salary
0  Alice   25  NYC   70000
1    Bob   30   LA   85000
25


In [8]:
# =========================================================
# BOOLEAN FILTERING - the most common way to query a DataFrame
# =========================================================

print(df[df["age"] > 28])                          # rows where age > 28
print(df[(df["age"] > 25) & (df["city"] == "NYC")])  # combine conditions with & (and) / | (or)
                                                      # NOTE: use & and |, not 'and'/'or', and
                                                      # wrap each condition in parentheses!

print(df[df["city"].isin(["NYC", "LA"])])            # membership test, like Python's 'in'
print(df.query("age > 28 and city == 'NYC'"))         # alternative, SQL-like syntax

      name  age city  salary
1      Bob   30   LA   85000
2  Charlie   35  NYC   62000
      name  age city  salary
2  Charlie   35  NYC   62000
      name  age city  salary
0    Alice   25  NYC   70000
1      Bob   30   LA   85000
2  Charlie   35  NYC   62000
      name  age city  salary
2  Charlie   35  NYC   62000


In [9]:
# =========================================================
# MODIFYING A DATAFRAME
# =========================================================

df2 = df.copy()   # always .copy() if you don't want to affect the original!

df2["bonus"] = df2["salary"] * 0.1                     # new column, computed from existing ones
df2["senior"] = df2["age"] >= 30                        # new boolean column
df2["initials"] = df2["name"].apply(lambda n: n[0])     # .apply() runs a function on every value

df2 = df2.rename(columns={"salary": "base_salary"})     # rename column(s)
df2 = df2.drop(columns=["bonus"])                        # drop a column
df2 = df2.sort_values("age", ascending=False)             # sort rows by a column

print(df2)


      name  age     city  base_salary  senior initials
2  Charlie   35      NYC        62000    True        C
1      Bob   30       LA        85000    True        B
3    Diana   28  Chicago        90000   False        D
0    Alice   25      NYC        70000   False        A


In [10]:
# =========================================================
# MISSING DATA - represented as NaN (Not a Number)
# =========================================================

df_missing = pd.DataFrame({
    "a": [1, 2, np.nan, 4],
    "b": [np.nan, 2, 3, 4],
})
print(df_missing)

print(df_missing.isna())              # boolean mask of missing values
print(df_missing.isna().sum())        # count of missing values PER COLUMN

print(df_missing.dropna())            # drop any row containing a NaN
print(df_missing.fillna(0))           # replace NaN with a specific value
print(df_missing.fillna(df_missing.mean()))   # replace NaN with each column's mean
print(df_missing.ffill())             # forward-fill: carry the last valid value forward


     a    b
0  1.0  NaN
1  2.0  2.0
2  NaN  3.0
3  4.0  4.0
       a      b
0  False   True
1  False  False
2   True  False
3  False  False
a    1
b    1
dtype: int64
     a    b
1  2.0  2.0
3  4.0  4.0
     a    b
0  1.0  0.0
1  2.0  2.0
2  0.0  3.0
3  4.0  4.0
          a    b
0  1.000000  3.0
1  2.000000  2.0
2  2.333333  3.0
3  4.000000  4.0
     a    b
0  1.0  NaN
1  2.0  2.0
2  2.0  3.0
3  4.0  4.0


In [11]:
# =========================================================
# GROUPBY - "split, apply, combine": group rows, then aggregate each group
# One of the most powerful and commonly used pandas features.
# =========================================================

sales = pd.DataFrame({
    "region": ["East", "East", "West", "West", "East"],
    "product": ["A", "B", "A", "B", "A"],
    "revenue": [100, 150, 200, 120, 90],
})

print(sales.groupby("region")["revenue"].sum())          # total revenue per region
print(sales.groupby("region")["revenue"].mean())         # average revenue per region
print(sales.groupby("region").agg({"revenue": ["sum", "mean", "count"]}))  # multiple aggregations
print(sales.groupby(["region", "product"])["revenue"].sum())   # group by MULTIPLE columns

region
East    340
West    320
Name: revenue, dtype: int64
region
East    113.333333
West    160.000000
Name: revenue, dtype: float64
       revenue                  
           sum        mean count
region                          
East       340  113.333333     3
West       320  160.000000     2
region  product
East    A          190
        B          150
West    A          200
        B          120
Name: revenue, dtype: int64


In [12]:
# =========================================================
# MERGING DATAFRAMES - like SQL joins
# =========================================================

employees = pd.DataFrame({
    "emp_id": [1, 2, 3],
    "name": ["Alice", "Bob", "Charlie"],
    "dept_id": [10, 20, 10],
})
departments = pd.DataFrame({
    "dept_id": [10, 20, 30],
    "dept_name": ["Engineering", "Sales", "Marketing"],
})

# how="inner"/"left"/"right"/"outer" - same semantics as SQL joins
print(pd.merge(employees, departments, on="dept_id", how="inner"))
print(pd.merge(employees, departments, on="dept_id", how="left"))

# concat - stack DataFrames on top of each other (or side by side with axis=1)
more_employees = pd.DataFrame({"emp_id": [4], "name": ["Diana"], "dept_id": [20]})
print(pd.concat([employees, more_employees], ignore_index=True))

   emp_id     name  dept_id    dept_name
0       1    Alice       10  Engineering
1       2      Bob       20        Sales
2       3  Charlie       10  Engineering
   emp_id     name  dept_id    dept_name
0       1    Alice       10  Engineering
1       2      Bob       20        Sales
2       3  Charlie       10  Engineering
   emp_id     name  dept_id
0       1    Alice       10
1       2      Bob       20
2       3  Charlie       10
3       4    Diana       20


In [13]:
# =========================================================
# .apply() and .map() - custom row/column-wise transformations
# =========================================================

df3 = df.copy()

# .map() -> element-wise, works on a Series
df3["city_short"] = df3["city"].map({"NYC": "NY", "LA": "LA", "Chicago": "CHI"})

# .apply() on a Series -> element-wise function
df3["age_group"] = df3["age"].apply(lambda a: "young" if a < 30 else "senior")

# .apply() on a DataFrame with axis=1 -> row-wise, gets the whole row each time
df3["summary"] = df3.apply(lambda row: f"{row['name']} ({row['age']}) - {row['city']}", axis=1)

print(df3)

      name  age     city  salary city_short age_group               summary
0    Alice   25      NYC   70000         NY     young      Alice (25) - NYC
1      Bob   30       LA   85000         LA    senior         Bob (30) - LA
2  Charlie   35      NYC   62000         NY    senior    Charlie (35) - NYC
3    Diana   28  Chicago   90000        CHI     young  Diana (28) - Chicago


In [14]:
# =========================================================
# PIVOT TABLES - reshape data, similar to an Excel pivot table
# =========================================================

sales = pd.DataFrame({
    "date": ["2024-01", "2024-01", "2024-02", "2024-02"],
    "region": ["East", "West", "East", "West"],
    "revenue": [100, 200, 150, 250],
})

pivot = sales.pivot_table(index="date", columns="region", values="revenue", aggfunc="sum")
print(pivot)

# melt - the OPPOSITE of pivot: turn columns back into rows ("unpivot" / "long format")
melted = pivot.reset_index().melt(id_vars="date", var_name="region", value_name="revenue")
print(melted)

region   East  West
date               
2024-01   100   200
2024-02   150   250
      date region  revenue
0  2024-01   East      100
1  2024-02   East      150
2  2024-01   West      200
3  2024-02   West      250


In [15]:
# =========================================================
# TIME SERIES - pandas has strong built-in datetime support
# =========================================================

dates = pd.date_range("2024-01-01", periods=5, freq="D")   # 5 consecutive days
ts = pd.Series([10, 12, 9, 15, 11], index=dates)
print(ts)

ts_df = pd.DataFrame({"value": [10, 12, 9, 15, 11]}, index=dates)
print(ts_df.resample("2D").mean())    # resample to a 2-day frequency, averaging within each bucket

# parsing date strings
df_dates = pd.DataFrame({"date": ["2024-01-01", "2024-02-15"], "value": [1, 2]})
df_dates["date"] = pd.to_datetime(df_dates["date"])   # convert str -> datetime
df_dates["month"] = df_dates["date"].dt.month           # .dt accessor -> extract date parts
print(df_dates)

2024-01-01    10
2024-01-02    12
2024-01-03     9
2024-01-04    15
2024-01-05    11
Freq: D, dtype: int64
            value
2024-01-01   11.0
2024-01-03   12.0
2024-01-05   11.0
        date  value  month
0 2024-01-01      1      1
1 2024-02-15      2      2


In [16]:
# =========================================================
# CLEANUP - remove demo files this notebook created
# =========================================================
import os
for filename in ("people.csv", "people.json"):
    if os.path.exists(filename):
        os.remove(filename)
print("Cleaned up demo files.")


Cleaned up demo files.
